# Unclipped MSE maps and vertical cost profiles

Use **Runtime → Change runtime type → CPU**, then **Run all**. This viewer reads the numeric heatmaps already saved by the multi-band experiment in Google Drive. It runs no simulation or training and needs no repository clone or model checkpoints.

The default source is `MyDrive/datasets/curriculum_cost_adapter/multiband_curriculum_v1`. Change `ROOT` below if your folder differs. New PNG/PDF figures are saved under `diagnostics/cost_profiles`; set `SAVE_FIGURES=False` to display only. Existing experiment files are read without modification.


## 1. Load the exact saved costs

Authorize Drive access. Available versions are discovered automatically; missing v3 is expected when band 3 collected no successful episodes. The table reports how many points exceeded the old color ceiling. The new ceiling covers the largest raw or adapted value across **both recipes and all saved versions**, preserving direct comparisons without saturation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/datasets/curriculum_cost_adapter/multiband_curriculum_v1')
SAVE_FIGURES = True
FIGURES = ROOT / 'diagnostics' / 'cost_profiles'
RECIPES = ('original', 'clean')
BANDS = ((8, 14, 'band 3'), (14, 22, 'band 2'), (22, 30, 'band 1'), (30, 36, 'near'))
GOAL = (50, 42)

# Read the exact saved numbers, with no encoder, world model or adapter loading.
maps, rows = {}, []
reference_points = reference_raw = reference_signature = None
for recipe in RECIPES:
    maps[recipe] = {}
    for version in range(4):
        path = ROOT / f'branches/{recipe}/heatmaps/v{version}.json'
        if not path.exists():
            print(f'{recipe} v{version}: no saved map; omitted')
            continue
        commit_path = path.with_name(f'v{version}.commit.json')
        if not commit_path.exists():
            raise ValueError(f'Missing completed-map record: {commit_path}')
        commit = json.loads(commit_path.read_text())
        payload = path.read_bytes()
        if hashlib.sha256(payload).hexdigest() != commit['data_sha256']:
            raise ValueError(f'Saved numeric map hash differs: {path}')
        data = json.loads(payload)
        if data['recipe'] != recipe or data['version'] != version:
            raise ValueError(f'Wrong recipe/version: {path}')
        if any(data.get(key) != value for key, value in commit['inputs'].items()):
            raise ValueError(f'Map provenance differs: {path}')
        points = np.asarray(data['points'], dtype=float)
        raw = np.asarray(data['raw'], dtype=float)
        adapted = np.asarray(data['adapted'], dtype=float)
        if (points.ndim != 2 or points.shape[1] != 2 or raw.shape != (len(points),)
                or adapted.shape != raw.shape or not np.isfinite(points).all()
                or not np.isfinite(raw).all() or not np.isfinite(adapted).all()
                or (raw < 0).any() or (adapted < 0).any()):
            raise ValueError(f'Invalid saved numeric map: {path}')
        if reference_points is None:
            reference_points, reference_raw, reference_signature = points, raw, data['signature']
        elif (not np.array_equal(points, reference_points)
              or not np.array_equal(raw, reference_raw) or data['signature'] != reference_signature):
            raise ValueError(f'Maps do not share the same grid/raw costs/goal/runtime: {path}')
        maps[recipe][version] = adapted
        rows.append(dict(recipe=recipe, version=f'v{version}',
            raw_max=float(raw.max()), adapted_max=float(adapted.max()),
            previous_display_max=data['display_max'],
            above_previous_scale=int((adapted > data['display_max']).sum())))
if reference_points is None:
    raise FileNotFoundError(f'No saved heatmap JSON files found under {ROOT}')
xs = np.unique(reference_points[:, 0])
ys = np.unique(reference_points[:, 1])
expected = np.array([(x, y) for y in ys for x in xs])
if not np.array_equal(expected, reference_points):
    raise ValueError('Expected the complete saved rectangular grid, ordered by y then x')
RAW = reference_raw.reshape(len(ys), len(xs))
for recipe in RECIPES:
    maps[recipe] = {v: a.reshape(RAW.shape) for v, a in maps[recipe].items()}
maximum = max([float(RAW.max())] + [float(a.max()) for versions in maps.values() for a in versions.values()])
DISPLAY_MAX = maximum * 1.03
print(f'Expanded shared color/cost limit: {DISPLAY_MAX:.6f}; maximum saved cost: {maximum:.6f}')
print('v0 = inherited adapter; v1 = after band 1; v2 = after band 2; v3 = after band 3, if saved.')
print(f"{'recipe':<10} {'version':<8} {'raw max':>10} {'adapted max':>13} {'old ceiling':>13} {'above old ceiling':>19}")
for row in rows:
    print(f"{row['recipe']:<10} {row['version']:<8} {row['raw_max']:>10.6f} "
          f"{row['adapted_max']:>13.6f} {row['previous_display_max']:>13.6f} "
          f"{row['above_previous_scale']:>15}/{len(reference_points)}")

def finish(figure, name):
    if SAVE_FIGURES:
        FIGURES.mkdir(parents=True, exist_ok=True)
        figure.savefig(FIGURES / f'{name}.png', dpi=170, bbox_inches='tight')
        figure.savefig(FIGURES / f'{name}.pdf', bbox_inches='tight')
    plt.show()
    plt.close(figure)


## 2. Heatmaps with the full cost range

The maps below all use the same expanded scale, starting at zero. No values exceed its upper limit. The star marks the goal `(50,42)`; horizontal dashed lines mark the band boundaries. Training overlays are omitted so the numeric field is easier to see.

Look for whether costs fall smoothly as you move upward toward the goal, and whether a lower-cost pocket remains near the bottom. A higher cost far from the goal is not itself a problem; the ordering of nearby positions matters.


In [ ]:
for recipe in RECIPES:
    if not maps[recipe]:
        continue
    panels = [('Raw', RAW)] + [(f'Adapter v{v}', a) for v, a in sorted(maps[recipe].items())]
    fig, axes = plt.subplots(1, len(panels), figsize=(3.1 * len(panels), 6.2),
                             sharex=True, sharey=True, squeeze=False, constrained_layout=True)
    for ax, (label, values) in zip(axes[0], panels):
        image = ax.pcolormesh(xs, ys, values, shading='nearest', cmap='viridis',
                             vmin=0, vmax=DISPLAY_MAX)
        for lo, hi, band in BANDS:
            ax.axhline(lo, color='white', linestyle='--', linewidth=0.7, alpha=0.7)
        ax.axhline(36, color='white', linestyle='--', linewidth=0.7, alpha=0.7)
        ax.scatter(*GOAL, marker='*', s=100, c='orange', edgecolors='black', linewidths=0.6)
        ax.set(title=f'{label}\nmax = {values.max():.4f}', xlabel='Position x (simulation units)',
               aspect='equal', xlim=(xs.min()-1, xs.max()+1), ylim=(ys.min()-1, ys.max()+1))
    axes[0, 0].set_ylabel('Position y (simulation units)')
    fig.colorbar(image, ax=axes[0].tolist(), shrink=0.8, label='Latent MSE — same expanded scale for every map')
    fig.suptitle(f'{recipe}: full saved cost range — no color saturation', fontsize=13)
    finish(fig, f'{recipe}_unclipped_maps')


## 3. Vertical cost profiles

Each panel fixes **x** and plots cost against **y**. The horizontal axis is now position y, not time. Read from left to right: the position moves upward toward goal y=42. Points are the actual saved grid measurements, joined by straight lines; intermediate positions were not measured.

- **Falling curve:** moving toward the goal lowers this image cost.
- **Rising curve:** moving toward the goal raises this image cost. This is a potential misleading preference to investigate.
- **Flat curve:** this vertical movement changes the cost little.

At x=50 the path aligns with the goal; at the other x values the path only reaches the goal's y-coordinate. These are real-image diagnostics at fixed x. They do not test wall feasibility, velocity, or the world model's predicted candidate costs, so a rising section is a clue rather than proof of the cause of failed planning.


In [ ]:
# Each curve holds x fixed and moves vertically through real grid observations.
# Stop at goal y: beyond y=42, increasing y moves away from the goal again.
requested_x = (40, 46, 50, 54)
x_indices = [int(np.argmin(np.abs(xs-x))) for x in requested_x]
if len(set(x_indices)) != len(x_indices):
    raise ValueError('Saved grid cannot supply four distinct requested x profiles')
keep = ys <= GOAL[1]
profile_y = ys[keep]
styles = {0: ('#0072b2', '-'), 1: ('#d55e00', '--'),
          2: ('#009e73', '-.'), 3: ('#cc79a7', ':')}
for recipe in RECIPES:
    if not maps[recipe]:
        continue
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True,
                             constrained_layout=True)
    for ax, column in zip(axes.flat, x_indices):
        for index, (lo, hi, band) in enumerate(BANDS):
            ax.axvspan(lo, hi, color='0.5', alpha=0.10 if index % 2 == 0 else 0.025)
            ax.axvline(lo, color='0.6', linestyle=':', linewidth=0.7)
            ax.text((lo+hi)/2, DISPLAY_MAX * 0.97, band, ha='center', va='top', fontsize=9)
        ax.plot(profile_y, RAW[keep, column], color='black', linewidth=2,
                marker='o', markersize=3, label='Raw')
        for version, values in sorted(maps[recipe].items()):
            color, linestyle = styles[version]
            ax.plot(profile_y, values[keep, column], color=color, linestyle=linestyle,
                    marker='.', markersize=5, linewidth=1.7, label=f'Adapter v{version}')
        ax.axvline(GOAL[1], color='0.4', linestyle='--', linewidth=1)
        ax.set(title=f'Fixed x = {xs[column]:g}', xlim=(profile_y.min(), GOAL[1]),
               ylim=(0, DISPLAY_MAX), xlabel='Position y → toward goal y=42', ylabel='Latent MSE')
        ax.grid(axis='y', alpha=0.18)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='outside lower center', ncol=len(labels), frameon=False)
    fig.suptitle(f'{recipe}: cost along vertical paths\n'
                 'Read left → right: falling = lower cost toward goal; rising = higher cost toward goal', fontsize=12)
    finish(fig, f'{recipe}_vertical_profiles')
if SAVE_FIGURES:
    print('Saved PNG and PDF figures:', FIGURES)


## What to return

Download this executed notebook (**File → Download → Download .ipynb**) and attach it here. The full-range maps and profiles will then let us examine actual numeric slopes rather than the saturated yellow display. No additional experiment run is needed.
